# Section 2.7 Embeddings

The last step in preparing the input text for LLM training is to convert the token IDs into embedding vectors. As a preliminary step, we must initialize GPT-like decoder-only transformer.  A continuous vector representation, or embedding, is necessary since GPT-like LLMs are deep neural networks trained with the backpropagation algorithm.  Let’s see how the token ID to embedding vector conversion works with a hands-on example. Suppose we have the following four input tokens with IDs 2, 3, 5, and 1:

In [7]:
import torch

# Reuse this in later modules instead of scattering `.cuda()` calls.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# For standalone tensors, create them directly on the selected device.
input_ids = torch.tensor([2, 3, 5, 1], device=device)

print(input_ids)


tensor([2, 3, 5, 1])


For the sake of simplicity, suppose we have a small vocabulary of only 6 words (instead of the 50,257 words in the BPE tokenizer vocabulary), and we want to create embeddings of size 3 (in GPT-3, the embedding size is 12,288 dimensions):

In [8]:
vocab_size = 6
output_dim = 3

Using the vocab_size and output_dim, we can instantiate an embedding layer in PyTorch, setting the random seed to 123 for reproducibility purposes:

In [9]:
torch.manual_seed(123)

# Move modules to the selected device so their weights match GPU input tensors.
embedding_layer = torch.nn.Embedding(vocab_size, output_dim).to(device)
print(embedding_layer.weight)


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


The weight matrix of the embedding layer contains small, random values. These values are optimized during LLM training as part of the LLM optimization itself. Moreover, we can see that the weight matrix has six rows and three columns. There is one row for each of the six possible tokens in the vocabulary, and there is one column for each of the three embedding dimensions.

Essentially, we've randomly placed each token in the vocabulary (all 6 of them) randomly into a 3 dimensional embedding space.

Now, let’s apply it to a token ID to obtain the embedding vector:

In [ ]:
print(embedding_layer(torch.tensor([3], device=device)))

print(embedding_layer(input_ids))     # does you'd expect


tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


If we compare the embedding vector for token ID 3 to the previous embedding matrix, we see that it is identical to the fourth row (Python starts with a zero index, so it’s the row corresponding to index 3). In other words, the embedding layer is essentially a lookup operation that retrieves rows from the embedding layer’s weight matrix via a token ID.